# NB-11-CAPACITY · Jupyter companion

Calculations use the private Workbench API, pinned to version **1.43.0**. You need your individual API key and the private input bundle supplied for **NB-11-CAPACITY**.

1. Open this notebook in a fresh Python 3.12 or 3.13 kernel. Keep `notebook-methods-v1.43.0.zip` beside it when working locally; otherwise setup downloads the checksum-verified public helper archive.
2. Run the helper setup cell, then select your private input directory or ZIP. Local Jupyter can use `EWB_NOTEBOOK_INPUT_DIR` or `EWB_NOTEBOOK_INPUT_ZIP`. Colab uploads send inputs to your Google runtime; use local Jupyter when the inputs must stay on your computer.
3. Run the teaching cells in order. Inspect inputs, receipts, tables and plots locally. Results are saved beneath `data/processed/notebook-results`; save the notebook privately after execution.

The download contains no saved execution outputs. [Previously reviewed figures and results](https://sherandoresearch.github.io/power-markets-series/NB-11-CAPACITY.html) remain available in the static reader. A successful calculation validates its numerical contract, not a scientific claim. Saved results retain their original version; these notebooks request version 1.43.0 explicitly.

The static reader preserves previously reviewed results; rerunning this notebook creates new private execution evidence.

<!-- book-volume-context -->
## Where this fits in the volumes

Read Volume V §11.4 first. The volume defines an electricity-interconnector utilization field and requires a declared direction, denominator, and missing-value rule. This notebook transfers that method to retained ENTSOG gas-transport observations: physical flow divided by firm technical capacity for the same point, operator, direction, and complete hour. The volume’s IFA example uses directional available capacity; this notebook deliberately defines a different gas field using firm technical capacity, in kWh/h of gas energy.

The economic question is how fully booked transport capacity can coexist with low physical use. Bookings reserve rights; measured flow records realized transport. The notebook also shows why a zero denominator remains undefined and why collector receipt time is not source publication time. It is an observed-data extension of the representation chapters, not an electricity capacity-market model, a DC-OPF/shadow-price calculation, or evidence that utilization predicts power prices.

- [Volume V, §11.4 — Derived fields and chosen definitions (start here)](https://sherandoresearch.github.io/power-markets-series/vol5_representing_power_markets.html#c11-4)
- [Volume V, §8.2 — The mapping contract: identity, intervals, units, and direction](https://sherandoresearch.github.io/power-markets-series/vol5_representing_power_markets.html#c8-2)
- [Volume V, §6.6 — Publication time and receipt time](https://sherandoresearch.github.io/power-markets-series/vol5_representing_power_markets.html#c6-6)

In [ ]:
# Install the reviewed transport and notebook helpers. This cell makes no API calls.
import hashlib, importlib.metadata, io, json, os, pathlib, ssl, stat, subprocess, sys, tempfile, urllib.request, zipfile

if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Use a Python 3.12 or 3.13 kernel for this reviewed release.")
helper_name = "notebook-methods-v1.43.0.zip"
expected_sha256 = "2bea8f592cae108b96369caded46f15631a60f1cd22d033cdd46905c3ce0ee18"
expected_files = json.loads('{"requirements.txt": "d3057422eeabe167fdc242f31c5038add6e0aa9aa73fb31227eb92b688cbb553", "src/ewb_methods_client/__init__.py": "faddf128f002243598cb56ab227945e50a2c6219453f24a2c633bf32c7ccf4ab", "src/methods_notebooks/__init__.py": "94e70a66bebea6221748a48192079f14297c552697aebc78205fd49128f9a345", "src/methods_notebooks/session.py": "86f788c33cc0db110ae1ae41f5cd4588af7029d8cd3aa935e491cabf6af40b3e", "src/methods_notebooks/setup.py": "17990abbd57a1472a44071834a7364604b452694605e6e4b19624ef8377e6d9f"}')
helper_path = pathlib.Path(os.environ.get("EWB_NOTEBOOK_HELPERS_ZIP", helper_name)).expanduser()
if helper_path.is_file():
    helper_bytes = helper_path.read_bytes()
else:
    helper_url = "https://github.com/sherandoresearch/power-markets-notebooks/releases/download/notebooks-v1.43.0/" + helper_name
    # Only the public, checksum-pinned transport helpers are downloaded here.
    # Inputs and API credentials are selected in the next cell.
    # Python.org macOS installs may lack a system CA bundle. Use the verified
    # certifi trust store when present; certificate and hostname checks stay on.
    try:
        import certifi
    except ImportError:
        tls_context = ssl.create_default_context()
    else:
        tls_context = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(helper_url, timeout=90, context=tls_context) as response:
        helper_bytes = response.read(1_000_001)
if len(helper_bytes) > 1_000_000 or hashlib.sha256(helper_bytes).hexdigest() != expected_sha256:
    raise ValueError("Notebook helper release checksum mismatch")
with zipfile.ZipFile(io.BytesIO(helper_bytes)) as helper_zip:
    members = helper_zip.infolist()
    if len(members) != len(expected_files) or {m.filename for m in members} != set(expected_files):
        raise ValueError("Unexpected helper archive contents")
    if any(stat.S_IFMT(m.external_attr >> 16) not in (0, stat.S_IFREG) or
           m.flag_bits & 1 or m.file_size > 100_000 for m in members):
        raise ValueError("Unsupported helper archive member")
    helper_contents = {name: helper_zip.read(name) for name in expected_files}
if any(hashlib.sha256(value).hexdigest() != expected_files[name] for name, value in helper_contents.items()):
    raise ValueError("Notebook helper member checksum mismatch")
requirements = helper_contents["requirements.txt"].decode().splitlines()
missing = []
for requirement in requirements:
    package, version = requirement.split("==")
    try:
        current_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        current_version = None
    if current_version != version:
        if package in sys.modules:
            raise RuntimeError("Restart in a fresh kernel before installing " + requirement)
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
for requirement in requirements:
    package, version = requirement.split("==")
    if importlib.metadata.version(package) != version:
        raise RuntimeError("Dependency version mismatch: " + package)
if any(name == "methods_notebooks" or name.startswith("methods_notebooks.") or
       name == "ewb_methods_client" or name.startswith("ewb_methods_client.") for name in sys.modules):
    raise RuntimeError("Restart the kernel before rerunning helper setup.")
helper_parent = pathlib.Path.cwd() / "data/processed/notebook-helpers"
helper_parent.mkdir(parents=True, exist_ok=True, mode=0o700)
helper_root = pathlib.Path(tempfile.mkdtemp(prefix="v1.43.0-", dir=helper_parent))
for name, value in helper_contents.items():
    target = helper_root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(value)
sys.path.insert(0, str(helper_root / "src"))
del helper_bytes, helper_contents
print("Notebook helpers verified. Calculations use Workbench 1.43; inputs and results stay private.")


In [ ]:
from methods_notebooks.setup import configure_notebook
setup_status = configure_notebook('NB-11-CAPACITY', require_key=True)
print(setup_status)


# NB-11-CAPACITY — Physical use and booked capacity
**Question:** when is flow divided by capacity meaningful? This companion to
NB-11-ENTSOG uses the same real DIS-00191 / BE-TSO-0001 hourly flow panel,
adding retained technical, available, booked and interruptible intervals.
The window is three gas days, 10–12 October 2026 (04:00 UTC to 04:00 UTC).
No synthetic observations are created. Prerequisites: interval joins, units,
and the distinction between physical throughput and contractual rights.
We audit the denominator, calculate a bounded utilization ratio, and preserve
zero-denominator cases. Reruns need private API access and retained inputs.

In [ ]:
NOTEBOOK_ID='NB-11-CAPACITY'

import copy, json
from datetime import datetime
from types import SimpleNamespace
import numpy as np
from methods_notebooks.session import Session, MethodsError, setup_plot
session = Session(NOTEBOOK_ID, version='1.43.0')
inputs = session.inputs
responses, execution_plan = {}, []
def execute(name, method, request, parents=()):
    request = copy.deepcopy(request)
    if parents:
        request['parent_runs'] = [{k:responses[p]['receipt'][k] for k in
            ('run_id','scientific_method_id','method_version','result_sha256')} for p in parents]
    response = session.execute(method, request)
    responses[name] = response
    execution_plan.append(dict(name=name,method=method,request=request,parents=list(parents),rest_receipt=response['receipt']))
    (session.output/'execution-plan.json').write_text(json.dumps(execution_plan,indent=2,allow_nan=False)+'\n')
    return response['result']
def reject(method, request):
    try:
        session.execute(method, request)
    except MethodsError as exc:
        assert str(exc) == 'Methods request refused (HTTP 422)'
        print('Expected remote rejection:',method,str(exc))
    else:
        raise AssertionError('Invalid retained-input mutation accepted')
plt=setup_plot()

from collections import Counter
from datetime import timedelta
print('Local raw capacity inspection:',inputs['capacity_records'][:2])
print('Local raw flow inspection:',inputs['raw_flow']['data'][:2])
snapshot=inputs['raw_flow']['meta']['snapshot_id'];audit={}
for feed in ('capacity-technical','capacity-available','firm-booked','interruptible'):
    latest=inputs['raw_capacity_captures'][feed]['structuredContent']
    prior=inputs['raw_capacity_captures']['asof_'+feed]['structuredContent']
    assert latest['meta']['snapshot_id']==prior['meta']['snapshot_id']==snapshot
    rows=latest['data'];assert rows==prior['data']
    keys=[tuple(r[k] for k in ('indicator','point_key','direction','operator_key','valid_from_utc')) for r in rows]
    assert len(set(keys))==len(keys)
    assert all(datetime.fromisoformat(r['retrieved_utc'])<=datetime.fromisoformat(inputs['collector_cutoff']) for r in prior['data'])
    audit[feed]=dict(rows=len(rows),collector_asof_equal=True)
c=dict(directions={},audit=audit,narrow_query_rows=len(inputs['raw_capacity_captures']['narrow_technical']['structuredContent']['data']))
for direction in ('entry','exit'):
    request=inputs['requests'][direction]
    result=execute(direction,'capacity_utilization',request)['utilization_result']
    records=request['inputs']['flow_capacity_records']['numerators']
    expected=[datetime.fromisoformat(inputs['collector_cutoff'])+timedelta(hours=i) for i in range(72)]
    assert [datetime.fromisoformat(r['start']) for r in records]==expected
    assert [r['numerator_assertion'] for r in result['rows']]==[r['assertion_id'] for r in records]
    ledger={(r['numerator_assertion'],r['capacity_kind']):r for r in result['selection_ledger']}
    assert len(ledger)==len(result['selection_ledger'])==432
    assert all(r['full_interval_covered'] for r in ledger.values())
    values={label:np.asarray([ledger[r['assertion_id'],kind]['selected_value'] for r in records]) for label,kind in inputs['capacity_kind_mapping'].items()}
    c['directions'][direction]=dict(times=expected,flow=np.asarray([r['value'] for r in records]),values=values,
        ratio=np.asarray([r['ratio'] for r in result['rows']],dtype=float),
        overlap_hours=sum(any(ledger[r['assertion_id'],kind]['eligible_count']>1 for kind in inputs['capacity_kind_mapping'].values()) for r in records),
        status=dict(Counter(inputs['normalized_assertion_sources'][r['assertion_id']]['flow_status'] for r in records)))
entry=c['directions']['entry'];exit=c['directions']['exit']
print(json.dumps(c['audit'],indent=2))
for direction,d in c['directions'].items():print(direction,{'hours':len(d['flow']),'flow_status':d['status'],'overlap_hours':d['overlap_hours']})

## 1. Interval containment, not equality on a date
The four capacity captures contain 33 intervals, including historical periods.
Match point, operator and direction, retaining indicator identity. A capacity
interval must contain the entire flow hour: `valid_from ≤ hour_start` and
`hour_end ≤ valid_to`. The snapshot matches the retained flows:
`entsog-20261015T122505Z-cfb5bf43`. Units are kWh/h throughout; an hourly
integral is kWh of gas energy. Dividing by 1,000 gives MWh of gas,
not electrical output; conversion to electricity requires a plant model.

The API filters capacity by interval **start**, not overlap. A retained query
with the flow window's lower start returns zero technical-capacity rows. That
does not mean capacity is zero or unknown: the relevant interval began earlier.
Query the selected entity without a lower start bound, then perform containment.
For overlapping intervals the descriptor recommends the newest start; ambiguous
equal-start candidates must fail. No overlaps occur in this sample.

In [ ]:
print('Technical rows when incorrectly applying the flow lower bound:',c['narrow_query_rows'])
assert c['narrow_query_rows']==0
for direction,d in c['directions'].items():
    print(direction,{k:np.unique(v).tolist() for k,v in d['values'].items()})
    assert d['overlap_hours']==0

## 2. A positive exit denominator and an undefined entry ratio
Exit technical capacity is 44,925,471 kWh/h. The 72 observed flow rates range
from 4,088,560 to 12,281,260 kWh/h. Utilization is the reported flow divided by
technical capacity for the same direction and full hour. It is descriptive,
not a measure of available capacity for a proposed transaction.
Entry has 72 zero flows and 72 zero capacities. Its 0/0 ratio is undefined;
neither zero filling nor an epsilon denominator is justified.
All exit flow rows are **Provisional**; all entry rows are Confirmed. The exit
percentages describe this retained vintage and may change with revisions.

In [ ]:
assert np.isnan(entry['ratio']).all()
assert np.isfinite(exit['ratio']).all()
print('Exit utilization min / mean / max (%):',np.round(100*np.array([exit['ratio'].min(),exit['ratio'].mean(),exit['ratio'].max()]),3).tolist())
print('Entry undefined hours:',int(np.isnan(entry['ratio']).sum()))
plt=setup_plot();fig,axes=plt.subplots(2,1,figsize=(11,6),sharex=True)
axes[0].plot(exit['times'],exit['flow']/1e6,color='#88C0D0',label='Provisional physical flow')
axes[0].plot(exit['times'],exit['values']['Firm Technical']/1e6,color='#EBCB8B',label='Firm technical capacity')
axes[0].set(ylabel='Million kWh/h',title='Exit: same point, operator, direction and hour');axes[0].legend()
axes[1].plot(exit['times'],100*exit['ratio'],color='#A3BE8C')
axes[1].set(ylabel='Utilization (%)',xlabel='UTC hour beginning',title='Exit utilization; entry is undefined (0/0)')
for ax in axes:ax.grid(alpha=.3)
fig.autofmt_xdate();fig.tight_layout();plt.show()

## 3. Fully booked does not mean fully used
Exit firm booked capacity equals technical capacity, while reported firm
available and interruptible quantities are zero. Technical − booked − available
reconciles here, but that is an observed sample identity, not a universal rule.
Physical flow averages about 14.39% of technical capacity despite full booking.
Capacity rights and actual transport measure different things. Zero available
capacity must not be used as the utilization denominator or read as an outage.

We keep entry and exit separate and do not sum different operators' reports.
This point's label and a single operator's records do not establish a network
aggregate. No attribution to individual shippers or causal explanation follows.

In [ ]:
v=exit['values']
np.testing.assert_allclose(v['Firm Technical']-v['Firm Booked']-v['Firm Available'],0)
fig,ax=plt.subplots(figsize=(10,4))
labels=['Technical','Firm booked','Firm available','Mean physical flow']
values=[v['Firm Technical'].mean(),v['Firm Booked'].mean(),v['Firm Available'].mean(),exit['flow'].mean()]
bars=ax.bar(labels,np.array(values)/1e6,color=['#EBCB8B','#B48EAD','#D8DEE9','#88C0D0'])
ax.bar_label(bars,fmt='%.2f',padding=4,color='#ECEFF4');ax.set_ylim(0,52)
ax.set(ylabel='Million kWh/h',title='Reported exit rights versus provisional physical use');fig.tight_layout();plt.show()

## 4. Capacity known at a cutoff is not a forecast of flow
Four retained collector-as-of requests at 10 October 04:00 UTC return the same
interval histories as latest. Their receipts do not exceed that cutoff. The
subsequent flow outcomes are retrospective; this is not a predictive backtest.
Source-published as-of is unsupported. Withdrawals to NA and interval tombstones
are incompletely represented, so matching latest and as-of cannot prove perfect
reconstruction of the source's historical state.

The 72 hours have constant denominators; the mean hourly percentage equals
total flow energy divided by total technical-capacity energy. With changing
capacity, report the intended weighting rather than assuming those means agree.

In [ ]:
assert all(a['collector_asof_equal'] for a in c['audit'].values())
energy_ratio=exit['flow'].sum()/v['Firm Technical'].sum()
np.testing.assert_allclose(energy_ratio,exit['ratio'].mean())
print('Three-day flow / capacity energy (%):',round(100*energy_ratio,3))

## Exercise and next evidence
Why does a zero result from the narrow capacity query not authorize filling
missing denominators with zero? The query excluded intervals that started
earlier. Why can 100% booked capacity coexist with 14% physical utilization?
Rights reserve capacity; realized transport need not use it all.
Broader conclusions require additional observed points, revised flow vintages,
and verified overlap/withdrawal semantics. This notebook supplies one bounded
case, with provisional outcomes and undefined entry ratios kept visible.
Source: retained EWB ENTSOG descriptors, complete query captures, and the
existing NB-11-ENTSOG flow receipt. No fresh API request runs inside the notebook.

In [ ]:
bad=copy.deepcopy(inputs['requests']['exit']);bad['inputs']['utilization_definition']['ratio']='power_flow_available';reject('capacity_utilization',bad)
bad=copy.deepcopy(inputs['requests']['exit']);bad['inputs']['flow_capacity_records']['capacities']=[r for r in bad['inputs']['flow_capacity_records']['capacities'] if r['kind']!='firm_technical_capacity'];reject('capacity_utilization',bad)
def plain(value):
    if isinstance(value,np.ndarray):return np.where(np.isnan(value),None,value).tolist() if np.issubdtype(value.dtype,np.number) else value.tolist()
    if isinstance(value,datetime):return value.isoformat()
    raise TypeError(type(value))
(session.output/'consumer-summary.json').write_text(json.dumps(c,default=plain,allow_nan=False,indent=2)+'\n')
summary=session.finish(dict(observed_only=True,full_hour_containment=True,zero_denominators_undefined=True,
    provisional_flow_disclosed=True,all_33_original_intervals=True,all_144_flow_hours=True,all_864_capacity_selection_checks=True,
    source_history_certified=False,predictive_backtest=False,new_synthetic_data=False),dict(selection_policy=inputs['selection_policy']))
print(summary)